# Notebook 3 — Temporal Data Splitting

## 1. Load Labeled Data

In [2]:
import pandas as pd
from pathlib import Path

input_path = Path("../data/artifacts/labeled_table.csv")

ml_table = pd.read_csv(input_path)

print("Shape:", ml_table.shape)

Shape: (99441, 21)


## 2. Convert Purchase Timestamp

In [3]:
ml_table["order_purchase_timestamp"] = pd.to_datetime(
    ml_table["order_purchase_timestamp"],
    errors="coerce"
)

print(
    ml_table["order_purchase_timestamp"].min(),
    "→",
    ml_table["order_purchase_timestamp"].max()
)

2016-09-04 21:15:19 → 2018-10-17 17:30:18


## 3. Remove Unknown Labels

In [4]:
model_data = ml_table.dropna(subset=["late"]).copy()

model_data["late"] = model_data["late"].astype(int)

print("Original rows:", len(ml_table))
print("Rows with known label:", len(model_data))
print("Dropped unknown label:", len(ml_table) - len(model_data))

Original rows: 99441
Rows with known label: 96476
Dropped unknown label: 2965


## 4. Sort Data by Purchase Time

In [5]:
model_data = model_data.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

print(model_data[
    [
        "order_id",
        "order_purchase_timestamp",
        "late"
    ]
].head())

                           order_id order_purchase_timestamp  late
0  bfbd0f9bdef84302105ad712db648a6c      2016-09-15 12:16:38     1
1  3b697a20d9e427646d92567910af6d57      2016-10-03 09:44:50     0
2  be5bc2f0da14d8071e2d45451ad119d9      2016-10-03 16:56:50     0
3  65d1e226dfaeb8cdc42f665422522d14      2016-10-03 21:01:41     0
4  a41c8759fbe7aab36ea07e038b2d4465      2016-10-03 21:13:36     0


## 5. Define Temporal Split

In [6]:
n = len(model_data)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

print("Total:", n)
print("Train:", train_end)
print("Validation:", val_end - train_end)
print("Test:", n - val_end)

Total: 96476
Train: 67533
Validation: 14471
Test: 14472


## 6. Create Train, Validation and Test Sets

In [7]:
train = model_data.iloc[:train_end].copy()

validation = model_data.iloc[train_end:val_end].copy()

test = model_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 21)
Validation: (14471, 21)
Test: (14472, 21)


## 7. Check Date Ranges

In [8]:
print("Train:")
print(train["order_purchase_timestamp"].min(), "→", train["order_purchase_timestamp"].max())

print("\nValidation:")
print(validation["order_purchase_timestamp"].min(), "→", validation["order_purchase_timestamp"].max())

print("\nTest:")
print(test["order_purchase_timestamp"].min(), "→", test["order_purchase_timestamp"].max())

Train:
2016-09-15 12:16:38 → 2018-04-15 20:07:56

Validation:
2018-04-15 20:10:23 → 2018-06-21 07:50:39

Test:
2018-06-21 08:29:29 → 2018-08-29 15:00:37


## 8. Check Label Distribution

In [9]:
for name, df in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{name}")
    print(df["late"].value_counts())
    print("Late %:", round(df["late"].mean() * 100, 2))


Train
late
0    61436
1     6097
Name: count, dtype: int64
Late %: 9.03

Validation
late
0    13698
1      773
Name: count, dtype: int64
Late %: 5.34

Test
late
0    13515
1      957
Name: count, dtype: int64
Late %: 6.61


## 9. Check for Data Leakage

In [10]:
train_ids = set(train["order_id"])
validation_ids = set(validation["order_id"])
test_ids = set(test["order_id"])

print("Train ∩ Validation:", len(train_ids & validation_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(validation_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 10. Save Data Splits

In [11]:
from pathlib import Path

output_dir = Path("../data/artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

train.to_csv(output_dir / "train.csv", index=False)
validation.to_csv(output_dir / "validation.csv", index=False)
test.to_csv(output_dir / "test.csv", index=False)

print("Saved successfully!")
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Saved successfully!
Train: 67533
Validation: 14471
Test: 14472
